In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from lightgbm import LGBMClassifier

In [2]:
train_df = pd.read_csv("/kaggle/input/competitions/playground-series-s6e9/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/playground-series-s6e9/test.csv")

In [3]:
train_df.head()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No
2,2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes
3,3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
4,4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,No


In [4]:
X = train_df.drop(columns=['id','Will_Buy_EV'])
Y = train_df['Will_Buy_EV']

In [5]:
cat_cols = X.select_dtypes(include='object').columns
num_cols = X.select_dtypes(exclude='object').columns

In [6]:
print(cat_cols)
print(num_cols)

Index(['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible',
       'Subsidy_Available', 'Range_Anxiety_Level'],
      dtype='object')
Index(['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned',
       'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work',
       'Environmental_Concern_Level'],
      dtype='object')


In [7]:
cat_pipeline = Pipeline([
    ('LabelEncoder',OrdinalEncoder()),
])

preprocessor = ColumnTransformer([
    ('categoric',cat_pipeline,cat_cols)
],remainder='passthrough'
)

In [8]:
model = Pipeline([
    ('preprocessor',preprocessor),
    ('lgbm',LGBMClassifier(learning_rate=0.1, n_estimators=100, random_state=42))
])

In [9]:
model.fit(X,Y)

[LightGBM] [Info] Number of positive: 116779, number of negative: 551886
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015039 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 619
[LightGBM] [Info] Number of data points in the train set: 668665, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.174645 -> initscore=-1.553058
[LightGBM] [Info] Start training from score -1.553058


/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('categoric',
                                                  Pipeline(steps=[('LabelEncoder',
                                                                   OrdinalEncoder())]),
                                                  Index(['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible',
       'Subsidy_Available', 'Range_Anxiety_Level'],
      dtype='object'))])),
                ('lgbm', LGBMClassifier(random_state=42))])

In [10]:
y_preds = model.predict_proba(test_df.drop(columns=['id']))[:,1]

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [11]:
submission = pd.DataFrame({
    'id': test_df['id'],
    'Will_Buy_EV':y_preds
})

submission.to_csv('submission.csv', index=False)